# Ungraded Lab: Class Activation Maps with Fashion MNIST (PyTorch)

In this lab, you will see how to implement a simple class activation map (CAM) of a model trained on the [Fashion MNIST dataset](https://github.com/zalandoresearch/fashion-mnist). This will show what parts of the image the model was paying attention to when deciding the class of the image. Let's begin!

> This notebook is a PyTorch port of the original TensorFlow/Keras lab. The overall flow (data → classifier → CAM) is the same, but the model is an `nn.Module`, training uses an explicit loop, and the "CAM model" is built by slicing the network instead of creating a second Keras `Model`.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.datasets import FashionMNIST
from torchinfo import summary

# use a GPU (CUDA or Apple MPS) when available
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

## Download and Prepare the Data

In [ ]:
# load the Fashion MNIST dataset (downloaded to ./data on first run)
train_set = FashionMNIST(root="data", train=True, download=True)
test_set = FashionMNIST(root="data", train=False, download=True)

# torchvision keeps the raw arrays as uint8 tensors of shape (N, 28, 28)
X_train, Y_train = train_set.data.numpy(), train_set.targets.numpy()
X_test, Y_test = test_set.data.numpy(), test_set.targets.numpy()

In [ ]:
# Put an additional axis for the channels of the image.
# PyTorch uses channels-first (N, C, H, W), so the channel axis goes right after the batch axis.
# Fashion MNIST is grayscale so we place 1 there. Other datasets will need 3 if it's in RGB.
X_train = X_train.reshape(60000, 1, 28, 28)
X_test = X_test.reshape(10000, 1, 28, 28)

# Normalize the pixel values from 0 to 1
X_train = X_train / 255
X_test = X_test / 255

# Cast to float32 (the dtype PyTorch layers expect)
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

In [ ]:
def show_img(img):
    '''
    Utility function for reshaping and displaying an image.

    Args:
      img (array) -- a single Fashion MNIST image, of 784 values in any shape
    '''

    # convert to float array if img is not yet preprocessed
    img = np.array(img, dtype='float')

    # remove channel dimension
    img = img.reshape((28, 28))

    # display image
    plt.imshow(img)

In [ ]:
# test the function for the first train image. you can vary the index of X_train
# below to see other images

show_img(X_train[1])

## Build the Classifier

Let's quickly recap how we can build a simple classifier with this dataset.

### Define the Model

You can build the classifier with the model below. The image will go through 4 convolutions followed by pooling layers. The final `Linear` layer will output the *logits* for each class.

Note a couple of PyTorch conventions:
- `padding=1` with a 3x3 kernel is the equivalent of Keras' `padding='same'`, recovering the lost border pixels.
- Activations are separate modules (`nn.ReLU`) instead of an argument of the conv layer.
- `GlobalAveragePooling2D` becomes `nn.AdaptiveAvgPool2d(1)` followed by `nn.Flatten()`.
- The `softmax` is **not** part of the model. `nn.CrossEntropyLoss` applies `log_softmax` internally, so the model outputs raw logits and we apply `softmax` ourselves when we need probabilities.

In [ ]:
# use nn.Sequential (the closest equivalent of the Keras Sequential API)
model = nn.Sequential(
    # notice the padding parameter to recover the lost border pixels when doing the convolution
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    # pooling layer with a stride of 2 will reduce the image dimensions by half
    nn.MaxPool2d(kernel_size=2),

    # pass through more convolutions with increasing filters
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),

    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.ReLU(),

    # use global average pooling to take into account lesser intensity pixels
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),

    # output class logits (softmax is applied by the loss / at inference time)
    nn.Linear(128, 10),
).to(device)

summary(model, input_size=(1, 1, 28, 28), device=device)

### Train the Model

In [ ]:
# configure the training
loss_fn = nn.CrossEntropyLoss()   # equivalent of 'sparse_categorical_crossentropy'
optimizer = torch.optim.Adam(model.parameters())

# build tensor datasets and hold out 10% of the training data for validation (validation_split=0.1)
full_train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train).long())
n_val = len(full_train_ds) // 10
train_ds, val_ds = random_split(full_train_ds, [len(full_train_ds) - n_val, n_val])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)


def run_epoch(loader, model, loss_fn, optimizer, device, train):
    '''
    Runs one pass over `loader`, training or evaluating.

    Args:
      loader (DataLoader) -- yields (images, labels) batches
      model (nn.Module) -- classifier being trained or evaluated
      loss_fn (callable) -- loss applied to (logits, labels)
      optimizer (Optimizer) -- updates weights; only used when train is True
      device (torch.device) -- device the batches are moved to
      train (bool) -- True updates the weights, False only measures

    Returns:
      (float, float) -- mean loss and accuracy
    '''
    model.train(train)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(xb)
            correct += (logits.argmax(1) == yb).sum().item()
            count += len(xb)
    return total_loss / count, correct / count


# train the model. just run a few epochs for this test run. you can adjust later.
EPOCHS = 5
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, model, loss_fn, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(val_loader, model, loss_fn, optimizer, device, train=False)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
          f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

## Generate the Class Activation Map

To generate the class activation map, we want to get the features detected in the last convolution layer and see which ones are most active when generating the output probabilities. In our model above, we are interested in the layers shown below.

In [ ]:
# final convolution layer (its ReLU output is what Keras' Conv2D(activation='relu') would return)
print(model[9], model[10])

# global average pooling layer
print(model[11])

# output of the classifier
print(model[13])

You can now create your CAM model as shown below.

In Keras you would build a second `Model` with two outputs. In PyTorch, an `nn.Sequential` can simply be sliced: the first part gives the final convolution features and the remaining part gives the classifier output.

In [ ]:
class CAMModel(nn.Module):
    '''same as the previous model but with an additional output: the final conv features'''

    def __init__(self, model):
        '''
        Splits the trained classifier into a feature half and a classifier half.

        Args:
          model (nn.Sequential) -- the trained Fashion MNIST classifier
        '''
        super().__init__()
        self.features = model[:11]     # everything up to (and including) the last conv + ReLU
        self.classifier = model[11:]   # global average pooling -> flatten -> linear

    def forward(self, x):
        '''
        Runs the image through both halves of the split model.

        Args:
          x (tensor) -- batch of images, shape (N, 1, 28, 28)

        Returns:
          (tensor, tensor) -- last conv features (N, 128, 3, 3) and softmax probabilities (N, 10)
        '''
        features = self.features(x)
        logits = self.classifier(features)
        return features, torch.softmax(logits, dim=1)


cam_model = CAMModel(model).to(device).eval()
print(cam_model)

Use the CAM model to predict on the test set, so that it generates the features and the predicted probability for each class (`results`).

In [ ]:
# get the features and results of the test images using the newly created model
features_list, results_list = [], []
with torch.no_grad():
    for xb, in DataLoader(TensorDataset(torch.from_numpy(X_test)), batch_size=256):
        f, r = cam_model(xb.to(device))
        features_list.append(f.cpu())
        results_list.append(r.cpu())

# PyTorch gives features as (N, channels, height, width); move channels last so the
# CAM math below reads exactly like the Keras version: (N, height, width, channels)
features = torch.cat(features_list).permute(0, 2, 3, 1).numpy()
results = torch.cat(results_list).numpy()

# shape of the features
print("features shape: ", features.shape)
print("results shape", results.shape)

You can generate the CAM by getting the dot product of the class activation features and the class activation weights.

You will need the weights from the Global Average Pooling layer (GAP) to calculate the activations of each feature given a particular class.
- Note that you'll get the weights from the linear layer that follows the global average pooling layer.
  - The last conv layer has (h,w,depth) of (3 x 3 x 128), so there are 128 features.
  - The global average pooling layer collapses the h,w,f (3 x 3 x 128) into a vector of 128 values (1 per feature).
  - The activations from the global average pooling layer get passed to the last linear layer.
  - The last linear layer assigns weights to each of those 128 features (for each of the 10 classes),
  - So the weights of the last linear layer (which immediately follows the global average pooling layer) are referred to in this context as the "weights of the global average pooling layer".

For each of the 10 classes, there are 128 features, so there are 128 feature weights, one weight per feature.

In [ ]:
# these are the weights going into the softmax layer
last_dense_layer = model[-1]

# PyTorch stores Linear weights as (out_features, in_features) = (10, 128) and the biases as (10,)
print("last_dense_layer.weight has shape ", tuple(last_dense_layer.weight.shape))
print("last_dense_layer.bias has shape ", tuple(last_dense_layer.bias.shape))

# transpose to (features, classes) = (128, 10) to match the Keras layout used in the CAM code below
gap_weights = last_dense_layer.weight.detach().cpu().numpy().T

print(f"There are {gap_weights.shape[0]} feature weights and {gap_weights.shape[1]} classes.")

Now, get the features for a specific image, indexed between 0 and 999.

In [ ]:
# Get the features for the image at index 0
idx = 0
features_for_img = features[idx, :, :, :]

print(f"The features for image index {idx} has shape (height, width, num of feature channels) : ", features_for_img.shape)

The features have height and width of 3 by 3.  Scale them up to the original image height and width, which is 28 by 28.

In [ ]:
features_for_img_scaled = sp.ndimage.zoom(features_for_img, (28/3, 28/3, 1), order=2)

# Check the shape after scaling up to 28 by 28 (still 128 feature channels)
print("features_for_img_scaled up to 28 by 28 height and width:", features_for_img_scaled.shape)

For a particular class (0...9), get the 128 weights.

Take the dot product with the scaled features for this selected image with the weights.

The shapes are:
scaled features: (h,w,depth) of (28 x 28 x 128).
weights for one class: 128

The dot product produces the class activation map, with the shape equal to the height and width of the image: 28 x 28.

In [ ]:
# Select the weights that are used for a specific class (0...9)
class_id = 0
# take the dot product between the scaled image features and the weights for
gap_weights_for_one_class = gap_weights[:, class_id]

print("features_for_img_scaled has shape ", features_for_img_scaled.shape)
print("gap_weights_for_one_class has shape ", gap_weights_for_one_class.shape)
# take the dot product between the scaled features and the weights for one class
cam = np.dot(features_for_img_scaled, gap_weights_for_one_class)

print("class activation map shape ", cam.shape)

### Conceptual interpretation
To think conceptually about what what you're doing and why:
- In the 28 x 28 x 128 feature map, each of the 128 feature filters is tailored to look for a specific set of features (for example, a shoelace).  
  - The actual features are learned, not selected by you directly.
- Each of the 128 weights for a particular class decide how much weight to give to each of the 128 features, for that class.
  - For instance, for the "shoe" class, it may have a higher weight for the feature filters that look for shoelaces.
- At each of the 28 by 28 pixels, you can take the vector of 128 features and compare them with the vector of 128 weights.  
  - You can do this comparison with a dot product.
  - The dot product results in a scalar value at each pixel.
  - Apply this dot product across all of the 28 x 28 pixels.
  - The scalar result of the dot product will be larger when the image both has the particular feature (e.g. shoelace), and that feature is also weighted more heavily for the particular class (e.g shoe).

So you've created a matrix with the same number of pixels as the image, where the value at each pixel is higher when that pixel is relevant to the prediction of a particular class.

Here is the function that implements the Class activation map calculations that you just saw.

In [ ]:
def show_cam(image_index, features, results, gap_weights, images):
    '''
    Displays the class activation map of a particular image.

    Args:
      image_index (int) -- index of the image to show
      features (array) -- last-conv features for every test image, shape (N, 3, 3, 128)
      results (array) -- softmax probabilities for every test image, shape (N, 10)
      gap_weights (array) -- weights of the final Linear layer, shape (128, 10)
      images (array) -- the test images themselves, shape (N, 1, 28, 28)
    '''

    # takes the features of the chosen image
    features_for_img = features[image_index, :, :, :]

    # get the class with the highest output probability
    prediction = np.argmax(results[image_index])

    # get the gap weights at the predicted class
    class_activation_weights = gap_weights[:, prediction]

    # upsample the features to the image's original size (28 x 28)
    class_activation_features = sp.ndimage.zoom(features_for_img, (28/3, 28/3, 1), order=2)

    # compute the intensity of each feature in the CAM
    cam_output = np.dot(class_activation_features, class_activation_weights)

    print('Predicted Class = ' + str(prediction) + ', Probability = ' + str(results[image_index][prediction]))

    # show the upsampled image
    plt.imshow(np.squeeze(images[image_index], 0), alpha=0.5)

    # strongly classified (95% probability) images will be in green, else red
    if results[image_index][prediction] > 0.95:
        cmap_str = 'Greens'
    else:
        cmap_str = 'Reds'

    # overlay the cam output
    plt.imshow(cam_output, cmap=cmap_str, alpha=0.5)

    # display the image
    plt.show()

You can now test generating class activation maps. Let's use the utility function below.

In [ ]:
def show_maps(desired_class, num_maps, features, results, gap_weights, images):
    '''
    Goes through the first 10,000 test images and generates CAMs for the first
    `num_maps` images predicted as `desired_class`.

    Args:
      desired_class (int) -- class id to look for, 0 to 9
      num_maps (int) -- how many maps to display
      features, results, gap_weights, images -- as described in show_cam
    '''

    counter = 0

    if desired_class >= 10:
        print("please choose a class less than 10")
        return

    # go through the first 10000 images
    for i in range(0, 10000):
        # break if we already displayed the specified number of maps
        if counter == num_maps:
            break

        # images that match the class will be shown
        if np.argmax(results[i]) == desired_class:
            counter += 1
            show_cam(i, features, results, gap_weights, images)

For class 8 (handbag), you'll notice that most of the images have dark spots in the middle and right side.
- This means that these areas were given less importance when categorizing the image.
- The other parts such as the outline or handle contribute more when deciding if an image is a handbag or not.

Observe the other classes and see if there are also other common areas that the model uses more in determining the class of the image.

In [ ]:
show_maps(desired_class=7, num_maps=20, features=features, results=results,
          gap_weights=gap_weights, images=X_test)